# 1. Overview

This notebook runs the `gate` workflow as a thin operator console. It restores the required runtime inputs, runs the public `workflows.gate` workflow, reviews typed lifecycle outputs, and publishes receipt-verified artifacts to Drive.

Workflow logic lives in `text_to_sign_production.workflows.gate`; this notebook only configures, reviews, executes, and summarizes the public workflow contract.

# 2. Operator Configuration

Set the repository revision, runtime roots, requested splits, gates config path, and person-selection policy for this run. These values are reviewed before runtime setup or workflow execution.

## 2.1 Repository and roots

Configure the repository checkout and runtime/Drive roots used by setup, restore, and publish operations.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/0xmillennium/text-to-sign-production.git"
REPO_REF = "chore/core-layout-notebook-workflows"
PROJECT_ROOT = Path("/content/text-to-sign-production")
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/text-to-sign-production")

## 2.2 Gate workflow inputs

Configure the gate workflow inputs that are passed into the public workflow config.

In [ ]:
GATE_SPLITS = ("train", "val", "test")
GATES_CONFIG_RELATIVE_PATH = Path("configs/data/gates.yaml")
PERSON_SELECTION_POLICY = "highest_canonical_signal"

## 2.3 Configuration review

Review the selected values before preparing the runtime.

In [ ]:
print(f"Repository URL: {REPO_URL}")
print(f"Repository ref: {REPO_REF}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Drive project root: {DRIVE_PROJECT_ROOT}")
print(f"Gate splits: {GATE_SPLITS}")
print(f"Gates config: {GATES_CONFIG_RELATIVE_PATH}")
print(f"Person selection policy: {PERSON_SELECTION_POLICY}")

# 3. Bootstrap Boundary

This isolated prelude prepares the Colab environment only: mount Drive, ensure system tooling, check out the repository, install dependencies, and make `src` importable. Skip this section when the environment is already prepared. Workflow restore, execution, validation, and publish steps start later in the runtime console.

## 3.1 Mount Drive

Mount Google Drive and confirm the configured Drive parent exists.

In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
if not DRIVE_PROJECT_ROOT.parent.is_dir():
    raise FileNotFoundError(f"Drive MyDrive root is missing: {DRIVE_PROJECT_ROOT.parent}")
print(f"Drive mounted: {DRIVE_PROJECT_ROOT.parent}")

## 3.2 System packages

Ensure `zstd` is available for tar.zst restore and publish archive operations.

In [ ]:
import shutil

if shutil.which("zstd") is None:
    !sudo apt-get update
    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError("Failed to update apt package index.")

    !sudo apt-get install -y zstd
    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError("Failed to install zstd.")
    print("Installed zstd.")
else:
    print("zstd is already available.")

!zstd --version
if globals().get("_exit_code", 1) != 0:
    raise RuntimeError("zstd command is not available after preflight.")

## 3.3 Repository checkout

Clone the requested repository revision into the configured runtime project root.

In [ ]:
%cd /content

if PROJECT_ROOT.exists():
    print(f"Removing stale repository checkout: {PROJECT_ROOT}")
    !rm -rf {PROJECT_ROOT}
    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError(f"Failed to remove existing directory: {PROJECT_ROOT}")

!git clone {REPO_URL} {PROJECT_ROOT}
if globals().get("_exit_code", 1) != 0:
    raise RuntimeError("Failed to clone repository.")

!git -C {PROJECT_ROOT} checkout {REPO_REF}
if globals().get("_exit_code", 1) != 0:
    raise RuntimeError(f"Failed to checkout revision {REPO_REF}.")

print(f"Repository ready: {PROJECT_ROOT}")
!git -C {PROJECT_ROOT} rev-parse HEAD
if globals().get("_exit_code", 1) != 0:
    raise RuntimeError("Failed to determine checked out revision.")

## 3.4 Install dependencies

Install the Colab dependency set from the checkout. Colab may restart the runtime after package installation; if that happens, global variables are cleared, so rerun Operator Configuration and Runtime Setup cells through this point before continuing.

In [ ]:
%cd {PROJECT_ROOT}
%pip install --upgrade pip
%pip install -r "requirements-colab.txt"
print("Repository dependencies installed from requirements-colab.txt.")

## 3.5 Add source tree to import path

Run this after dependency installation and after any runtime restart. It depends on restored globals such as `PROJECT_ROOT`, so rerun the configuration cells first if Colab restarted.


In [ ]:
import sys

%cd {PROJECT_ROOT}
source_path = PROJECT_ROOT / "src"
if str(source_path) not in sys.path:
    sys.path.append(str(source_path))
print(f"Repository src directory available on sys.path: {source_path}")

## 3.6 Workflow API import

Import the public gate workflow API from `text_to_sign_production.workflows.gate`.


In [ ]:
from text_to_sign_production.workflows.foundation.review import display_review_sections
from text_to_sign_production.workflows.gate import (
    GateWorkflow,
    GateWorkflowConfig,
)

# 4. Runtime Console: Plan

Build the gate workflow object and inspect the compact runtime restore plan before any restore command runs.

## 4.1 Build workflow config

Create the public gate workflow config from the reviewed operator inputs.

In [ ]:
gate_config = GateWorkflowConfig(
    project_root=PROJECT_ROOT,
    drive_project_root=DRIVE_PROJECT_ROOT,
    splits=GATE_SPLITS,
    gates_config_relpath=GATES_CONFIG_RELATIVE_PATH,
    person_selection_policy=PERSON_SELECTION_POLICY,
)

## 4.2 Instantiate workflow

Instantiate the workflow facade that owns runtime planning, restore, processing, reports, and publish calls.

In [ ]:
gate = GateWorkflow(gate_config)
print("Gate workflow instantiated.")

## 4.3 Plan runtime

Build the runtime restore plan without executing it.

In [ ]:
gate_runtime_plan = gate.plan_runtime()
print("Gate runtime plan built.")

## 4.4 Review runtime plan

Review a compact operator summary of the runtime plan.

In [ ]:
display_review_sections(gate.review_runtime_plan(gate_runtime_plan))

# 5. Runtime Console: Restore and Verify

Validate the plan, restore runtime inputs, and verify readiness as separate lifecycle transitions.

## 5.1 Validate runtime plan

Validate the planned runtime operations against the workflow layout before restore.

In [ ]:
gate.validate_runtime_plan(gate_runtime_plan)
print("Runtime plan validated.")

## 5.2 Execute runtime restore

Run the reviewed restore operations through the workflow facade.

In [ ]:
gate_restore_result = gate.restore_runtime(gate_runtime_plan)
print("Gate runtime restore complete.")

## 5.3 Review restore result

Review the compact restore execution result.

In [ ]:
display_review_sections(gate.review_runtime_restore(gate_restore_result))

## 5.4 Verify runtime

Run the explicit runtime readiness checks after restore.

In [ ]:
gate_runtime_verification = gate.verify_runtime(gate_runtime_plan)
print(f"Runtime {gate_runtime_verification.readiness_level.value} checked.")

## 5.5 Review runtime verification

Review the compact readiness outcome and any failed check examples.

In [ ]:
display_review_sections(gate.review_runtime_verification(gate_runtime_verification))

# 6. Runtime Console: Process

Run gate processing and review only the compact processing outcome by default.

## 6.1 Execute processing

Execute gate processing after runtime readiness has been checked.

In [ ]:
gate_bundle = gate.execute_processing(
    gate_runtime_plan,
    gate_runtime_verification,
)
gate_result = gate_bundle.workflow_result
print("Gate workflow processing complete.")

## 6.2 Review compact processing summary

Review split counts, identity totals, and viability/drop totals without dumping every sample row.

In [ ]:
display_review_sections(gate.review_processing(gate_bundle))

# 7. Runtime Console: Outputs and Reports

Review planned outputs, write reports, and review compact artifact/final summaries before publishing.

## 7.1 Review planned outputs

Review planned output locations and counts without listing every payload artifact.

In [ ]:
display_review_sections(gate.review_outputs(gate_result))

## 7.2 Write reports

Materialize report files; detailed rows are written to report artifacts instead of displayed by default.

In [ ]:
gate_report_artifacts = gate.write_reports(gate_bundle)
print(f"Gate reports written: {gate_report_artifacts.index_json_path}")

## 7.3 Review written report artifacts

Review compact artifact counts and key report paths.

In [ ]:
display_review_sections(
    gate.review_written_artifacts(gate_bundle, gate_report_artifacts)
)

## 7.4 Review compact workflow summary

Review the compact workflow outcome before publishing.

In [ ]:
display_review_sections(
    gate.review_final_operator_summary(gate_bundle, gate_report_artifacts)
)

# 8. Runtime Console: Publish and Verify

Build, execute, verify, and review publish lifecycle states one step at a time.

## 8.1 Build publish plan

Build the publish plan from the processed bundle and written report artifacts.

In [ ]:
gate_publish_plan = gate.build_publish_plan(gate_bundle, gate_report_artifacts)
print("Gate publish plan built.")

## 8.2 Review compact publish plan

Review publish target and operation counts without dumping every target row.

In [ ]:
display_review_sections(gate.review_publish_plan(gate_publish_plan))

## 8.3 Execute publish

Run the publish plan through the workflow facade.

In [ ]:
gate_publish_execution = gate.execute_publish(gate_publish_plan)
print("Gate publish execution complete.")

## 8.4 Review compact publish execution

Review execution success/failure counts and failure examples only if present.

In [ ]:
display_review_sections(gate.review_publish_execution(gate_publish_execution))

## 8.5 Verify publish

Verify published targets after execution.

In [ ]:
gate_publish_verification = gate.verify_publish(gate_publish_plan)
print("Gate publish verification complete.")

## 8.6 Review compact publish verification

Review verification counts and failure examples without dumping every check row.

In [ ]:
display_review_sections(gate.review_publish_verification(gate_publish_verification))

## 8.7 Build publish result

Assemble the typed publish result after plan, execution, and verification are all available.

In [ ]:
gate_publish_result = gate.build_publish_result(
    gate_publish_plan,
    gate_publish_execution,
    gate_publish_verification,
)
print("Gate publish result built.")

## 8.8 Review compact publish result

Review the compact publish outcome for the operator console.

In [ ]:
display_review_sections(gate.review_publish_result(gate_publish_result))

# 9. Final Summary

Print one compact operator summary for workflow and publish outcomes. Detailed rows remain in report files and explicit detail review methods.

In [ ]:
display_review_sections(
    (
        *gate.review_final_operator_summary(gate_bundle, gate_report_artifacts),
        *gate.review_publish_result(gate_publish_result),
    )
)